# Persistence - I/O Devices Overview


## 🟢 Challenges
CPU and memory are relatively uniform:
- CPU executes instructions
- memory stores bytes

I/O is different because devices are wildly different in:
- purpose
- speed
- data format
- timing behavior
- reliability expectations
- control protocol

A keyboard, SSD, GPU, camera, and network card do not behave alike at all. The operating system has to make all of them look manageable.

So I/O is hard because it is really about communication with many kinds of external agents.

---

## 🟢 I/O is communication

A useful mental model is:

I/O = message exchange between the computer and devices

That communication may be:
- user ↔ device
    - Example: typing on a keyboard, moving a mouse, reading a display
- device ↔ device
    - Example: disk controller transferring data to memory, network adapter passing packets to host memory
- CPU/OS ↔ device controller
    - Example: “start read,” “status ready?,” “here is a buffer,” “operation finished”

So every device needs a protocol:
- what commands exist
- what status values mean
- how data is formatted
- what timing is allowed
- how errors are signaled

That is why the slide refers to an “agent-agent protocol.”

---

## 🟢 The four basic questions every device interface must answer

For any device, the system has to answer:
### What commands can I send?
- read sector
- write block
- reset device
- start DMA
- acknowledge interrupt
- etc

### What status can I read back?
- ready
- busy
- error
- data available
- transfer complete

### How is the data represented?
- bytes
- words
- blocks
- packets
- audio samples
- pixels

### How fast and with what timing constraints?
- keyboard is slow and bursty
- audio/video may require regular streaming
- storage is block-oriented
- networks are asynchronous and variable-rate

These differences are exactly why I/O software stacks become large and messy.

---

## 🟢 Device speed varies enormously

One big OS challenge is that device data rates differ by many orders of magnitude.

Roughly:
- keyboard: very slow
- mouse: slow
- audio: moderate streaming rate
- camera/video: much higher
- disk: block transfer oriented, medium to high
- network: can range from modest to extremely fast

This means the OS cannot use one simple strategy for all devices, such as:
- polling may be fine for a tiny embedded input source, but terrible for a fast disk
- buffering is critical for rate mismatch
- interrupts and DMA become important for high-throughput devices

So I/O design is really about managing mismatched speeds.

---

## 🟢 Devices are not connected directly “as raw gadgets”
A device is usually not exposed to the CPU directly. Instead it is presented through a device controller / interface.

That interface often contains:
- status register
- command register
- data register
- local controller logic
- sometimes local memory/buffers
- sometimes even its own processor or microcontroller

So when the OS “talks to a device,” it usually talks to the device controller registers, not the device mechanism itself, such as:
- writing a command register may tell a disk controller to begin a read
- reading a status register tells whether the device is ready or busy
- reading or writing a data register transfers data or control values

This is the meaning of “devices connected to system bus via interface ports.”

---

## 🟢 Why access to device registers must be protected

If any user program could freely access device registers, it could:
- crash the machine
- corrupt files
- spy on input/output
- interfere with other processes
- overwrite hardware state
- read sensitive data from devices or DMA buffers

So device access is typically restricted to:
- the kernel
- trusted drivers
- privileged instructions or mapped regions with protection

This is why the slide says device access must be protected.

---

## 🟢 Bus structure: not everything hangs off one identical wire

Modern systems typically have several interconnect layers.

A simplified view:
- CPU ↔ main memory on a fast memory path
- high-speed devices on a general I/O bus
- slower or more peripheral devices on peripheral buses

Examples of buses or interconnect families:
- memory bus / memory channel
- PCIe-like high-speed I/O
- SATA / USB / storage and peripheral buses

The exact hardware varies, but the idea is consistent:
- different parts of the machine communicate over different interconnects optimized for different jobs

This matters because the OS and device drivers often rely on bus properties:
- addressing
- transfer size
- interrupt delivery
- DMA support
- ordering rules

---

## 🟢 Device register access: two main models

The third slide introduces the two classic ways CPUs access device registers:
- Isolated I/O (separate I/O address space)
- Memory-mapped I/O (MMIO)

These are very important.

---

## 🟢 Isolated I/O / separate I/O address space

In this model:
- device ports live in a special I/O address space
- normal memory addresses are separate
- CPU uses special instructions to access device ports

On x86, classic examples are:
- IN
- OUT

Therefore:
- memory load/store instructions access RAM
- special I/O instructions access device ports

### What this means conceptually

The CPU has two “worlds” of addresses:
- memory addresses
- I/O port addresses
- And the hardware distinguishes them using:
- special instructions
- special control lines/signals

### Advantages
- clear separation between memory and devices
- device access is explicit
- can simplify some hardware distinctions

### Disadvantages
- requires special CPU instructions
- less uniform programming model
- compilers and languages do not naturally treat device registers like ordinary memory objects

---

## 🟢 Memory-mapped I/O (MMIO)

In MMIO:
- device registers are assigned ordinary memory addresses
- the CPU uses normal loads and stores
- no separate I/O instruction set is needed

So a read from a certain address may actually read:
- a NIC status register
- a GPU control register
- a UART data port
- even though it looks like a normal memory read at the instruction level.

The address decoder/hardware decides:
- this address goes to RAM
- that address goes to a device register

So the CPU just performs a load/store, and the interconnect routes it appropriately.

### Advantages
- uniform programming model
- ordinary load/store instructions work
- easier for compilers and many architectures
- no special I/O instruction requirement

### Disadvantages
- device registers consume part of the physical address space
- must handle caching carefully, because device registers are not normal RAM
- must preserve ordering semantics where needed

MMIO is extremely common in modern systems.

---

## 🟢 Comparing isolated I/O and memory-mapped I/O
### Isolated I/O
- separate address space
- special instructions like IN/OUT
- hardware explicitly distinguishes I/O operations

### Memory-mapped I/O
- shared address space with memory
- normal loads/stores access device registers
- simpler software model, very common today

### A good intuition:
isolated I/O says “devices are a separate kind of target”

MMIO says “devices look like special memory locations”

---

## 🟢 A simple end-to-end example
Suppose a process wants to read from disk.

At a high level:
- process calls read()
- kernel enters disk driver
- driver programs controller registers or DMA descriptors
- controller starts the transfer
- device/controller reads the block
- data is delivered to memory or buffers
- controller signals completion
- kernel wakes the waiting process

Even though the user sees one read call, underneath there is:
- register access
- bus transactions
- protocol handling
- memory/device coordination

That is why I/O feels much more complicated than ordinary CPU-memory execution.

---

## 🟢 I/O Methods
### The core problem:
A device is much slower and more irregular than the CPU.
So after the CPU asks a device to do something, what should happen next?

### A. Wait-loop I/O (busy waiting)
The CPU repeatedly checks the device until it is ready.

Typical pattern:
- read device status register
- if not ready, check again
- when ready, put/get data
- start next operation
- repeat

#### Advantage
- very easy to implement
- no interrupts needed
- useful for tiny operations or very simple hardware

#### Disadvantage
- CPU does no useful work while waiting. It burns cycles just checking

#### Best use cases
- simple embedded systems
- very short hardware waits
- early boot code
- tiny hardware-control loops

### B. Polling I/O
The OS periodically checks one or more devices to see whether they need service.

This is different from sitting forever on one device. Instead, the system may:
- check device A
- check device B
- check device C
- repeat

The “hub/tree” picture is showing a system where software walks through a set of devices looking for one that needs attention.

#### Polling is useful when:
- device events happen frequently
- the cost of interrupts would be too high
- the system wants predictable control over when checks happen

#### Disadvantage
- wasted checks when nothing has changed
- response time depends on polling frequency
- does not scale well if many devices are mostly idle

#### Typical use
- network processing in some high-performance systems
- USB-style enumeration/control logic
- systems where periodic checking is cheaper than many interrupts

### C. Interrupt-driven I/O
Instead of the CPU repeatedly checking the device, the device notifies the CPU when it needs attention or when an operation completes.
- CPU/driver starts I/O
- CPU does other work
- device finishes or needs service
- device raises an interrupt
- CPU temporarily stops current work
- interrupt handler runs
- OS/driver processes completion and resumes

#### Advantage

The CPU is not wasting time checking idle devices. It works on something else until the hardware says:
- “I’m done”
- “data is available”
- “an error occurred”

#### Disadvantage
Interrupts are not free:
- CPU must stop current execution
- save state
- enter kernel mode
- run handler
- restore state

So if interrupts happen too frequently, they can become overhead.

#### Best use cases
- devices that are usually idle but occasionally need attention
- disks, keyboards, mice, serial devices
- moderate-rate I/O completion signaling

### D. DMA (Direct Memory Access)
DMA is for data movement, not just notification.

Without DMA, the CPU may have to move data itself:
- device → CPU register
- CPU register → memory
- repeated many times

#### With DMA:
- the CPU sets up the transfer
- a DMA engine/controller moves the block directly between device and memory
- CPU is interrupted only at setup/completion points, not for every word or byte

#### Advantage
For large transfers, DMA dramatically reduces CPU overhead.

The CPU tells the DMA hardware something like:
- source device
- destination memory buffer
- transfer size
- direction
- start

Then DMA does the bulk copy.

#### Benefits
- much less CPU involvement
- better overlap of computation and I/O
- essential for high-throughput storage/network/graphics

#### Disadvantages
- extra hardware support
- cache coherence and memory-protection concerns
- more complex driver setup

#### Best use cases
- disks and SSDs
- network cards
- audio/video streams
- large block transfers